# RUNG-26b — de novo binder by **RFdiffusion** (the real external-key lever)

The field-proven method for hard, flat pMHC targets (AfDesign-hallucination, RUNG-26, got 0/6). Pipeline:
**Boltz folds the target pMHC** (proven in RUNG-26) → **RFdiffusion** generates binder backbones contacting the
mutated residue (hotspot) → **ProteinMPNN** designs sequences → **AF2 (ColabDesign)** validates (pae_interaction ≤ 10,
pLDDT ≥ 80) and scores discrimination vs WT.

**HONEST WARNING (read before running).** Claude cannot test GPU notebooks locally. The **RFdiffusion install
(Cell 4)** is the one untested-on-this-runtime part — it uses conda (`condacolab`, which RESTARTS the runtime) +
SE3Transformer, the heaviest install in this project. Expect a possible 1–2 debug rounds (like RUNG-26 took for the
*simpler* AfDesign). Cells are ordered so the irreplaceable target fold (Boltz) is saved to **Drive BEFORE** the
RFdiffusion install/restart, and the batch is time-boxed + resumable — so a crash never loses finished designs.

**Order:** Cell 1 clone → Cell 2 Boltz fold target→Drive (~20 min) → Cell 3 install ColabDesign (AF2 scorer) →
**Cell 4 install RFdiffusion (condacolab RESTART — the fragile cell)** → after restart re-run Cell 1 → Cell 5 SMOKE
(1 design end-to-end) → Cell 6 the batch (~4 h) → Cell 7 rank.  **Target:** IDH1-R132H/HLA-A\*01:01 (clean, no
natural TCR — RUNG-27a — so de novo design is the only route).

In [ ]:
#@title Cell 1 — clone + GPU + spec
import os, json, subprocess
from pathlib import Path
TARGET_ID = "IDH1_R132H_A0101" #@param {type:"string"}
REPO=Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
!nvidia-smi -L || print('!! NO GPU — Runtime>GPU (T4)')
SPEC=json.loads(subprocess.run(['python','scripts/52_binder_design.py','spec',TARGET_ID],capture_output=True,text=True).stdout)
globals().update(GROOVE=SPEC['mhc_groove'], PEP_MUT=SPEC['pep_mut'], PEP_WT=SPEC['pep_wt'], MUTPOS=SPEC['mut_residue_in_peptide'])
print('target:',SPEC['gene'],SPEC['mut_label'],SPEC['allele'],'| mut',PEP_MUT,'WT',PEP_WT,'| mut pep pos',MUTPOS)
print('[CELL 1] OK')

In [ ]:
#@title Cell 2 — fold MUT/WT pMHC with BOLTZ -> PDB on Drive (PROVEN, RUNG-20/26)  [~20 min, GPU]
import os, glob, subprocess, shutil, importlib.util as iu
from google.colab import drive; drive.mount('/content/drive')
WORK='/content/drive/MyDrive/cancer-recon/rung26b_rfdiff'; os.makedirs(WORK, exist_ok=True); os.environ['RF_WORK']=WORK
if iu.find_spec('boltz') is None: subprocess.run(['pip','install','-q','boltz'])
if iu.find_spec('gemmi') is None: subprocess.run(['pip','install','-q','gemmi'])
import gemmi
CACHE=f'{WORK}/boltz_cache'; os.makedirs(CACHE, exist_ok=True)
def fold_pmhc(kind, pep):
    pdb=f'{WORK}/pmhc_{kind}.pdb'
    if os.path.exists(pdb): print(f'  {kind}: cached'); return pdb
    y=f'{WORK}/pmhc_{kind}.yaml'; outd=f'{WORK}/boltz_{kind}'
    open(y,'w').write('sequences:\n  - protein:\n      id: A\n      sequence: '+GROOVE+'\n  - protein:\n      id: B\n      sequence: '+pep+'\n')
    r=subprocess.run(f'boltz predict {y} --use_msa_server --diffusion_samples 1 --cache {CACHE} --out_dir {outd}', shell=True, capture_output=True, text=True)
    print((r.stderr or '')[-800:])
    f=sorted(glob.glob(f'{outd}/**/*.pdb',recursive=True)) or sorted(glob.glob(f'{outd}/**/*.cif',recursive=True))
    if not f: print(f'  {kind}: NO STRUCTURE — paste stderr to Claude'); return None
    if f[0].endswith('.cif'): st=gemmi.read_structure(f[0]); st.setup_entities(); st.write_pdb(pdb)
    else: shutil.copy(f[0], pdb)
    return pdb
MUT_PDB=fold_pmhc('mut',PEP_MUT); WT_PDB=fold_pmhc('wt',PEP_WT)
# record the hotspot residue id (chain B, mutated position) for RFdiffusion
import gemmi as _g
def hotspot(pdb):
    st=_g.read_structure(pdb); chB=[c for c in st[0] if c.name=='B'][0]; return chB[MUTPOS-1].seqid.num
if MUT_PDB and WT_PDB:
    HOT=hotspot(MUT_PDB); os.environ['RF_HOT']=f'B{HOT}'; os.environ['RF_MUT_PDB']=MUT_PDB; os.environ['RF_WT_PDB']=WT_PDB
    open(f'{WORK}/meta.json','w').write(json.dumps({'mut_pdb':MUT_PDB,'wt_pdb':WT_PDB,'hotspot':f'B{HOT}','groove_len':len(GROOVE),'pep_len':len(PEP_MUT)}))
    print(f'[CELL 2] OK  hotspot=B{HOT}  (saved to Drive — survives the Cell-4 restart)')
else: print('[CELL 2] FAILED — do not proceed')

In [ ]:
#@title Cell 3 — install ColabDesign (the AF2 validator/scorer; torch-free jax) [~5 min]
import subprocess, importlib.util as iu, glob
if iu.find_spec('colabdesign') is None:
    subprocess.run(['pip','install','-q','git+https://github.com/sokrypton/ColabDesign.git@v1.1.1'])
if not glob.glob('params/params_model_*.npz'):
    subprocess.run('mkdir -p params && (aria2c -q -x16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar 2>/dev/null || wget -q https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar)', shell=True)
    subprocess.run('tar -xf alphafold_params_2022-12-06.tar -C params && rm -f alphafold_params_2022-12-06.tar', shell=True)
print('[CELL 3] OK (ColabDesign + AF2 params ready)')

In [ ]:
#@title Cell 4 — install RFdiffusion + ProteinMPNN  [FRAGILE — RESTARTS runtime; ~15-20 min]
#@markdown condacolab.install() RESTARTS the runtime. The Boltz target is already safe on Drive. After the
#@markdown restart: re-run Cell 1 (instant), then SKIP straight to Cell 5 (this cell's second half auto-detects).
import os, subprocess, importlib.util as iu
RF='/content/RFdiffusion'
if not os.path.exists(RF):
    subprocess.run(['git','clone','-q','https://github.com/RosettaCommons/RFdiffusion.git',RF])
    os.makedirs(f'{RF}/models', exist_ok=True)
    subprocess.run(f'wget -q http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt -O {RF}/models/Complex_base_ckpt.pt', shell=True)
    subprocess.run(['git','clone','-q','https://github.com/dauparas/ProteinMPNN.git','/content/ProteinMPNN'])
if iu.find_spec('condacolab') is None:
    subprocess.run(['pip','install','-q','condacolab'])
import condacolab
try:
    condacolab.check()      # already installed (post-restart)
    print('condacolab present — installing SE3 deps into base...')
    subprocess.run(f'cd {RF} && conda env update -n base -f env/SE3nv.yml -q', shell=True)
    subprocess.run(f'cd {RF}/env/SE3Transformer && pip install -q --no-cache-dir -r requirements.txt && python setup.py install', shell=True)
    subprocess.run(f'cd {RF} && pip install -q -e .', shell=True)
    print('[CELL 4] OK — RFdiffusion installed. Proceed to Cell 5 SMOKE.')
except Exception:
    print('Installing condacolab — runtime will RESTART. After it does: re-run Cell 1, then re-run THIS Cell 4.')
    condacolab.install()

In [ ]:
#@title Cell 5 — SMOKE: 1 design end-to-end (RFdiffusion -> ProteinMPNN -> AF2 score)  [~5-8 min, GPU]
import os, glob, subprocess, json
RF='/content/RFdiffusion'; WORK=os.environ['RF_WORK']; MUT_PDB=os.environ['RF_MUT_PDB']; HOT=os.environ['RF_HOT']
meta=json.load(open(f'{WORK}/meta.json')); GL=meta['groove_len']; PL=meta['pep_len']
CONTIG=f"[A1-{GL}/0 B1-{PL}/0 70-90]"            # keep MHC(A)+peptide(B) fixed; generate a 70-90aa binder
def rfdiffuse(out_prefix, n):
    cmd=(f"cd {RF} && python scripts/run_inference.py inference.input_pdb={MUT_PDB} "
         f"'contigmap.contigs={CONTIG}' 'ppi.hotspot_res=[{HOT}]' "
         f"inference.output_prefix={out_prefix} inference.num_designs={n}")
    r=subprocess.run(cmd, shell=True, capture_output=True, text=True); print((r.stderr or '')[-1200:]); return r.returncode
def mpnn(backbone_pdb):
    out=f'{WORK}/mpnn'; os.makedirs(out, exist_ok=True)
    cmd=(f"cd /content/ProteinMPNN && python protein_mpnn_run.py --pdb_path {backbone_pdb} "
         f"--pdb_path_chains B --out_folder {out} --num_seq_per_target 1 --sampling_temp 0.1 --seed 1 --batch_size 1")
    subprocess.run(cmd, shell=True, capture_output=True, text=True)
    fa=sorted(glob.glob(f'{out}/seqs/*.fa'))
    if not fa: return None
    seq=[l.strip() for l in open(fa[-1]) if l.strip() and not l.startswith('>')][-1]
    return seq.split('/')[-1] if '/' in seq else seq    # binder chain sequence
# AF2 score (reuse ColabDesign predict against the fixed target)
from colabdesign import mk_afdesign_model, clear_mem
import gemmi
def _hotnum(pdb):
    st=gemmi.read_structure(pdb); chB=[c for c in st[0] if c.name=='B'][0]; return f'B{chB[meta["pep_len"]*0+ (int(HOT[1:])- chB[0].seqid.num)].seqid.num}' if False else HOT
def score(target_pdb, seq):
    clear_mem(); m=mk_afdesign_model(protocol='binder', use_multimer=False, num_recycles=3, data_dir='.')
    m.prep_inputs(pdb_filename=target_pdb, chain='A,B', binder_len=len(seq), hotspot=HOT, rm_target_seq=False, ignore_missing=True)
    m.predict(seq=seq, verbose=False); log=m.aux['log']
    return {'pae_interaction':round(float(log['i_pae'])*31.0,3),'binder_plddt':round(float(log['plddt'])*100.0,1)}
rc=rfdiffuse(f'{WORK}/smoke/d', 1)
bb=sorted(glob.glob(f'{WORK}/smoke/*.pdb'))
print('RFdiffusion backbone:', bb[-1] if bb else 'NONE (paste stderr above to Claude)')
if bb:
    seq=mpnn(bb[-1]); print('ProteinMPNN seq:', seq)
    if seq:
        mut=score(os.environ['RF_MUT_PDB'], seq); wt=score(os.environ['RF_WT_PDB'], seq)
        print('MUT', mut, '| WT', wt, '| disc(Å)', round(wt['pae_interaction']-mut['pae_interaction'],2))
        print('[CELL 5] SMOKE OK — launch Cell 6' if 'pae_interaction' in mut else '[CELL 5] check')

In [ ]:
#@title Cell 6 — THE BATCH (RFdiffusion -> MPNN -> AF2, time-boxed + resumable)  [~4 h, GPU]
max_hours=4.0 #@param {type:'number'}
import os, glob, time, json
RF='/content/RFdiffusion'; WORK=os.environ['RF_WORK']; DES=f'{WORK}/designs'; os.makedirs(DES, exist_ok=True)
PAE_BIND=10.0; PLDDT_MIN=80.0
t_end=time.time()+max_hours*3600; i=len(glob.glob(f'{DES}/*/metrics.json'))
print(f'[CELL 6] resume from {i}; until', time.strftime('%H:%M',time.localtime(t_end)))
while time.time()<t_end:
    did=f'design_{i:04d}'; dd=f'{DES}/{did}'
    if os.path.exists(f'{dd}/metrics.json'): i+=1; continue
    os.makedirs(dd, exist_ok=True)
    try:
        rfdiffuse(f'{dd}/bb', 1); bb=sorted(glob.glob(f'{dd}/*.pdb'))
        if not bb: raise RuntimeError('no backbone')
        seq=mpnn(bb[-1])
        if not seq: raise RuntimeError('no mpnn seq')
        mut=score(os.environ['RF_MUT_PDB'], seq)
        rec={'design_id':did,'target':os.environ.get('TARGET_ID','IDH1_R132H_A0101'),'sequence':seq,'mut':mut}
        if mut['pae_interaction']<=PAE_BIND and mut['binder_plddt']>=PLDDT_MIN:
            rec['wt']=score(os.environ['RF_WT_PDB'], seq)
        json.dump(rec, open(f'{dd}/metrics.json','w'))
        print(f"  {did}: mut_pae={mut['pae_interaction']} plddt={mut['binder_plddt']}"+(f" DISC={round(rec['wt']['pae_interaction']-mut['pae_interaction'],2)}" if 'wt' in rec else ' (not a binder)'), flush=True)
    except Exception as e:
        print(f'  {did} FAILED: {e}', flush=True); json.dump({'design_id':did,'error':str(e)}, open(f'{dd}/metrics.json','w'))
    i+=1
print(f'[CELL 6] done — {i} attempted. Run Cell 7.')

In [ ]:
#@title Cell 7 — RANK + bundle
import os, sys, json, glob, zipfile, shutil
WORK=os.environ['RF_WORK']; DES=f'{WORK}/designs'; dst='runs/rung26b_rfdiff'; os.makedirs(dst, exist_ok=True)
for m in glob.glob(f'{DES}/*/metrics.json'):
    d=os.path.join(dst, os.path.basename(os.path.dirname(m))); os.makedirs(d, exist_ok=True); shutil.copy(m,d)
os.system(f'{sys.executable} scripts/52_binder_design.py rank {dst}')
b='/content/rung26b_rfdiff.zip'
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in glob.glob(f'{dst}/**/*',recursive=True):
        if os.path.isfile(x): z.write(x, arcname=os.path.relpath(x,'runs'))
print('[CELL 7] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(skipped:', e, ')')